# LLIE Colab L4 Pipeline

这份 notebook 面向 Colab `L4 GPU`，用于完成低光照增强论文的第一轮实验复现。

当前主线：

- `URetinex-Net` on `LOL-v1`
- `Diff-Retinex` on `LOL-v1`
- `Reti-Diff` on `LOLv2-Real` and `LOLv2-Synthetic`
- 自动记录状态、日志、指标与汇总表


In [1]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil

drive.mount('/content/drive')

# 按你的 Drive 实际路径修改
NOTEBOOK_ROOT = Path('/content/drive/MyDrive/thesis/experiments/colab_l4')
CONFIG_PATH = NOTEBOOK_ROOT / 'config' / 'run_config.json'
EXAMPLE_CONFIG = NOTEBOOK_ROOT / 'config' / 'run_config.example.json'

if not CONFIG_PATH.exists():
    shutil.copyfile(EXAMPLE_CONFIG, CONFIG_PATH)
    print(f'Created {CONFIG_PATH}. Edit it before continuing.')

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

print(json.dumps(cfg, ensure_ascii=False, indent=2))


Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/thesis/experiments/colab_l4/config/run_config.example.json'

In [ ]:
!nvidia-smi

REPOS_ROOT = Path(cfg['repos_root'])
OUTPUT_ROOT = Path(cfg['output_root'])
REPOS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Repos root:', REPOS_ROOT)
print('Output root:', OUTPUT_ROOT)


In [ ]:
!pip -q install pyyaml lpips scikit-image tensorboardX timm einops thop pyiqa openai-clip ftfy addict natsort lmdb facexlib


In [ ]:
import subprocess

repos = {
    'URetinex-Net': 'https://github.com/SZU-AdvTech-2023/262-URetinex-Net-Retinex-based-Deep-Unfolding-Network-for-Low-light-Image-Enhancement.git',
    'Diff-Retinex': 'https://github.com/XunpengYi/Diff-Retinex.git',
    'Reti-Diff': 'https://github.com/ChunmingHe/Reti-Diff.git',
}

for name, url in repos.items():
    target = REPOS_ROOT / name
    if target.exists():
        print(f'Skip clone, already exists: {target}')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=True)

subprocess.run(['pip', 'install', '-q', '-r', str(REPOS_ROOT / 'Reti-Diff' / 'requirements.txt')], check=True)
subprocess.run(['pip', 'install', '-q', '-e', str(REPOS_ROOT / 'Reti-Diff')], check=True)

print('Repositories are ready.')


In [ ]:
import subprocess

METHOD_DATASETS = {
    'uretinex': ['lol_v1'],
    'diff-retinex': ['lol_v1'],
    'reti-diff': ['lolv2_real', 'lolv2_syn'],
}

SCRIPT = NOTEBOOK_ROOT / 'scripts' / 'run_baseline.py'

for method, dataset_keys in METHOD_DATASETS.items():
    for dataset_key in dataset_keys:
        ds = cfg['datasets'][dataset_key]
        cmd = [
            'python', str(SCRIPT),
            '--method', method,
            '--output-root', str(OUTPUT_ROOT),
            '--dataset-name', dataset_key,
            '--lq-dir', ds['lq_dir'],
            '--gt-dir', ds['gt_dir'],
        ]

        if method == 'uretinex':
            cmd.extend(['--repo-root', str(REPOS_ROOT / 'URetinex-Net')])
        elif method == 'diff-retinex':
            cmd.extend([
                '--repo-root', str(REPOS_ROOT / 'Diff-Retinex'),
                '--diff-tdn-weight', cfg['weights']['diff_retinex']['tdn'],
                '--diff-rda-weight', cfg['weights']['diff_retinex']['rda'],
                '--diff-ida-weight', cfg['weights']['diff_retinex']['ida'],
            ])
        elif method == 'reti-diff':
            template_name = 'test_LLIE_real.yml' if dataset_key == 'lolv2_real' else 'test_LLIE_syn.yml'
            weight_name = 'llie_real' if dataset_key == 'lolv2_real' else 'llie_syn'
            cmd.extend([
                '--repo-root', str(REPOS_ROOT / 'Reti-Diff'),
                '--reti-template', str(REPOS_ROOT / 'Reti-Diff' / 'options' / template_name),
                '--reti-weight', cfg['weights']['reti_diff'][weight_name],
                '--reti-decom-weight', cfg['weights']['reti_diff']['retinex_decom'],
            ])

        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)


In [ ]:
import json
import subprocess

metrics_script = NOTEBOOK_ROOT / 'scripts' / 'compute_metrics.py'
meta_files = sorted(OUTPUT_ROOT.rglob('run_metadata.json'))

for meta_file in meta_files:
    meta = json.loads(meta_file.read_text(encoding='utf-8'))
    run_dir = meta_file.parent
    metrics_dir = run_dir / 'metrics'
    cmd = [
        'python', str(metrics_script),
        '--method', meta['method'],
        '--dataset-name', meta['dataset_name'],
        '--pred-dir', meta['pred_dir'],
        '--gt-dir', meta['gt_dir'],
        '--output-dir', str(metrics_dir),
    ]
    if meta.get('status_path'):
        cmd.extend(['--status-path', meta['status_path']])
    if meta['method'] == 'uretinex':
        cmd.extend(['--strip-suffix', '_URetinexNet'])
    if cfg['run_switches'].get('compute_metrics', True):
        print('Computing metrics:', ' '.join(cmd))
        subprocess.run(cmd, check=True)


In [ ]:
import subprocess

aggregate_script = NOTEBOOK_ROOT / 'scripts' / 'aggregate_results.py'
summary_root = OUTPUT_ROOT
aggregate_root = OUTPUT_ROOT / 'aggregated'
cmd = [
    'python', str(aggregate_script),
    '--input-root', str(summary_root),
    '--output-root', str(aggregate_root),
]
subprocess.run(cmd, check=True)

print((aggregate_root / 'aggregated_metrics.md').read_text(encoding='utf-8'))


## 后续建议

1. 第一轮先把这三条基线结果跑完整。
2. 记录 `status.json`、`run.log`、`summary.json` 到 Drive。
3. 再回到本地论文工作区，把均值指标与典型可视化填入第四章。
4. 如果后续还要扩展到 `SSP-IR`，建议单独开一个 notebook，不要和当前三条基线混在一起。
